In [3]:
import xarray as xr
import os
from pathlib import Path
import xarray as xr
import pandas as pd


# --- paths -------------------------------------------------------------------
working_dir = '/Users/jakobwerkgarner/code/mt_dsnow'
os.chdir(working_dir)



# Create Seasnoal data

use data from- > 1st of November to the 30th of April like done in Mag25

In [5]:
from pathlib import Path
import xarray as xr
import pandas as pd

# === Define paths ===
src = Path("calibration/calibration_data/raw_data/Mag25/SLF_dataset/Mag25_all.nc")
dst = Path("/Users/jakobwerkgarner/code/mt_dsnow/HNW_validation/validation_input")
dst.mkdir(parents=True, exist_ok=True)

# === Load dataset ===
ds = xr.open_dataset(src)

# Rename time → date
ds = ds.rename({"time": "date"})

# === Seasons (2016–2022) ===
seasons = [f"{y:02d}{(y+1)%100:02d}" for y in range(16, 22)]
for s in seasons:
    (dst / s).mkdir(parents=True, exist_ok=True)

# === Assign season per date ===
def get_season(ts):
    m, y = ts.month, ts.year
    if m in (11, 12):     # Nov, Dec ⇒ start in same year
        start, end = y, y + 1
    elif m in (1, 2, 3, 4):  # Jan–Apr ⇒ season is previous year
        start, end = y - 1, y
    else:
        return None
    return f"{start % 100:02d}{end % 100:02d}"

dates = pd.to_datetime(ds.date.values)
ds = ds.assign_coords(season=("date", [get_season(t) for t in dates]))

# Keep only valid seasons
ds = ds.where(ds.season.isin(seasons), drop=True)

# === Loop station × season ===
summary = []

for station in ds.station.values:
    ds_station = ds.sel(station=station)

    for s in seasons:
        sub = ds_station.where(ds_station.season == s, drop=True)

        # IMPORTANT FIX: use date instead of time
        if sub.date.size == 0:
            summary.append((station, s, 0))
            continue

        # Convert to DataFrame
        df = sub.to_dataframe().reset_index()

        # Remove coords not needed
        df = df.drop(columns=["station", "season"])

        # Convert date to YYYY-MM-DD string
        df["date"] = pd.to_datetime(df["date"]).dt.strftime("%Y-%m-%d")

        df.at[df.index[0], "HS"] = 0

        out_file = dst / s / f"{station}.csv"
        df.to_csv(out_file, index=False)

        summary.append((station, s, len(df)))

# === Save summary ===
summary_df = pd.DataFrame(summary, columns=["station", "season", "rows"])
summary_df.to_csv(dst / "season_station_summary.csv", index=False)

summary_df

,station,season,rows
0,Adelboden,1617,181
1,Adelboden,1718,181
2,Adelboden,1819,181
3,Adelboden,1920,182
4,Adelboden,2021,181
...,...,...,...
241,Zuoz,1718,181
242,Zuoz,1819,181
243,Zuoz,1920,182
244,Zuoz,2021,181
